# Module 7. Topic 13. Transformers: text summarization

The code of the second practical block of Topic 13 (fine-tuning T5 for review summarization).

**Important!** The full training takes a few hours: run this notebook on Kaggle ("Save Version" → "Save & Run All").
1. Select the accelerator "GPU T4 x2". GPU P100 is not supported by the current PyTorch build (CUDA 13 requires compute capability 7.5+).
2. Leave "Internet on": the dataset and the pre-trained model are downloaded by the code.

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/BetterED-Neoversity/DEEP-LEARNING-FOR-COMPUTER-VISION-AND-NLP/blob/main/notebooks/Module_7_Lecture_13_Class_summarization_kaggle.ipynb) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BetterED-Neoversity/DEEP-LEARNING-FOR-COMPUTER-VISION-AND-NLP/blob/main/notebooks/Module_7_Lecture_13_Class_summarization_kaggle.ipynb)

In [1]:
!pip install rouge_score
!pip install evaluate

In [2]:
import random

import numpy as np

from datasets import load_dataset

from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

from transformers import pipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import DataCollatorForSeq2Seq
from transformers import default_data_collator

import evaluate

import nltk
nltk.download('punkt_tab') # sentence tokenizer used to prepare texts for ROUGE

[nltk_data] Downloading package punkt_tab to /root/nltk_data...


True

In [3]:
device = "cuda" if torch.cuda.is_available() else None

## Data

In [4]:
raw_datasets = load_dataset("kartikay/review-summarizer")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['target', 'text'],
        num_rows: 793751
    })
})

The dataset is large (~800K reviews): for a training of reasonable duration we use a random 2% sample.

In [5]:
def sample_n_examples(dataset, p):
    n = int(len(dataset) * p)
    indices = random.sample(range(len(dataset)), n)
    return dataset.select(indices)

random.seed(42)
p = 0.02
# Sample N examples from each split in the DatasetDict
sampled_dataset_dict = {split: sample_n_examples(dataset, p) for split, dataset in raw_datasets.items()}

In [6]:
split_datasets = sampled_dataset_dict["train"].train_test_split(train_size=0.9, seed=20)
split_datasets["validation"] = split_datasets.pop("test")
split_datasets

DatasetDict({
    train: Dataset({
        features: ['target', 'text'],
        num_rows: 14287
    })
    validation: Dataset({
        features: ['target', 'text'],
        num_rows: 1588
    })
})

In [7]:
def show_samples(dataset, num_samples=3, seed=42):
    sample = dataset["train"].shuffle(seed=seed).select(range(num_samples))
    for example in sample:
        print(f"\n'>> Title: {example['target']}'")
        print(f"'>> Review: {example['text']}'")

show_samples(raw_datasets)


'>> Title: Can't repay my loan'
'>> Review: I am receiving daily calls to my workplace and personal cell phone everyday from Navient about my student loans that I can not repay at this time do to the outstanding balance and interest on the loans. Navient also calls my mother, grandmother, and husband daily. My mother and grandmother are co-signers on a few of my loans and are unable to help me repay my loans due to their own financial restraints. This is harassment plain and simple and calling my myself and my family members multiple times a day ( including Saturdays ) will not change the mere fact that i can not afford these ridiculous monthly payments they are requesting from me. 

XXXX XXXX University XXXX College XXXX'

'>> Title: Incorrect information on your report'
'>> Review: EQUIFAX consistently fails to list my XXXX and XXXX XXXX XXXX XXXX XXXX XXXX credit account onto my EQUIFAX credit report. EQUIFAX non compliance with the creditor lowers my credit score. I have reapetedl

## Model and tokenizer

In [8]:
model_checkpoint = "google-t5/t5-small"

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 7935.04it/s]

In [9]:
max_input_length = 128
max_target_length = 30

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=max_input_length,
        truncation=True
    )
    labels = tokenizer(
        examples["target"], max_length=max_target_length, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [10]:
tokenized_datasets = split_datasets.map(preprocess_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns(raw_datasets["train"].column_names)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14287
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1588
    })
})

## Metric

In [11]:
rouge_score = evaluate.load("rouge")

In [12]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

scores = rouge_score.compute(
    predictions=[generated_summary], references=[reference_summary]
)
scores

{'rouge1': np.float64(0.923076923076923),
 'rouge2': np.float64(0.7272727272727272),
 'rougeL': np.float64(0.923076923076923),
 'rougeLsum': np.float64(0.923076923076923)}

## Training

In [13]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [14]:
batch_size = 32
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=batch_size,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=batch_size
)

In [15]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [16]:
num_train_epochs = 10
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

In [17]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # ROUGE expects a newline after each sentence
    preds = ["\n".join(nltk.sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(nltk.sent_tokenize(label)) for label in labels]

    return preds, labels

In [18]:
output_dir = "results-t5-finetuned"

In [19]:
progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for step, batch in enumerate(train_dataloader):
        outputs = model(**batch.to(device))
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            generated_tokens = model.generate(
                batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                max_new_tokens=max_target_length, # summaries as long as the reference ones
            )
            labels = batch["labels"].to(device)

            generated_tokens = generated_tokens.cpu().numpy()
            labels = labels.cpu().numpy()

            # Replace -100 in the labels as we can't decode them
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            if isinstance(generated_tokens, tuple):
                generated_tokens = generated_tokens[0]
            decoded_preds = tokenizer.batch_decode(
                generated_tokens, skip_special_tokens=True
            )
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

            decoded_preds, decoded_labels = postprocess_text(
                decoded_preds, decoded_labels
            )

            rouge_score.add_batch(predictions=decoded_preds, references=decoded_labels)

    # Compute metrics
    result = rouge_score.compute()
    # Extract the median ROUGE scores
    result = {key: value * 100 for key, value in result.items()}
    result = {k: round(float(v), 4) for k, v in result.items()}
    print(f"Epoch {epoch}:", result)

    # Save and upload
    model.save_pretrained(output_dir)

100%|██████████| 4470/4470 [12:58<00:00,  6.85it/s]

Epoch 0: {'rouge1': 11.132, 'rouge2': 2.791, 'rougeL': 10.89, 'rougeLsum': 10.8778}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.82it/s]

Epoch 1: {'rouge1': 14.6709, 'rouge2': 5.3415, 'rougeL': 14.5332, 'rougeLsum': 14.5349}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.50it/s]

Epoch 2: {'rouge1': 16.597, 'rouge2': 7.5299, 'rougeL': 16.5143, 'rougeLsum': 16.4728}


... (11 outputs left out)

Epoch 8: {'rouge1': 20.5355, 'rouge2': 11.0706, 'rougeL': 20.3848, 'rougeLsum': 20.3384}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.43it/s]

Epoch 9: {'rouge1': 21.0959, 'rouge2': 11.4907, 'rougeL': 20.9546, 'rougeLsum': 20.8957}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.31it/s]

## Summaries generated by the fine-tuned model

In [20]:
import textwrap

# Load the fine-tuned model saved by the training loop
trained_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(device)
trained_model.eval()

# Reviews from the validation set: the model has not seen them during training.
# A few rows of the dataset have the review and its title swapped, so we take
# the first 5 examples in which the review is longer than its title
validation = split_datasets["validation"].select(range(20))
examples = [example for example in validation if len(example["text"]) > len(example["target"])][:5]

for example in examples:
    inputs = tokenizer(example["text"], max_length=max_input_length, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        summary_ids = trained_model.generate(**inputs, max_new_tokens=max_target_length)
    print("Review:   ", textwrap.shorten(example["text"], width=300, placeholder=" ..."))
    print("Reference:", example["target"])
    print("Generated:", tokenizer.decode(summary_ids[0], skip_special_tokens=True))
    print()

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 8486.04it/s]

Review:    Time for my daily complaint with Equifax. I would like to state again that the accounts from XXXX XXXX XXXX, XXXX XXXX XXXX, and XXXX XXXX are not mine and are fradulent. XXXX XXXX XXXX claims that a name unknown to me was on this account and it was charged off with a person using my wife 's ...
Reference: Incorrect information on credit report
Generated: Incorrect information on your report

Review:    Our pediatrician recommended Pediasure for our 2 year old son since he was underweight. He'll go days with hardly eating a thing and doesn't seem to mind. It's very stressful trying to make sure he has a healthy diet. I refuse to go the junk food route just to get food in him. But he needed 1-2 ...
Reference: Awesome nutritional supplement
Generated: Getting a healthy diet

Review:    These Honey Flavored Hard Pieces tasted okay if you like honey. Actually, I used them when I had a sore throat and thought the honey filled center soothed my throat for a time. But, as far as gi